# Manual Soft Prompt Tuning for Vision Language Models

This notebook implements manual soft prompt tuning for VLMs on the RSICD dataset. Supports easy switching between:

- Qwen 2.5 7B VL
- Gemma3 4B (text-only)
- Pixtral 12B
- Phi 3.5 Vision 4.2B

by [Gayanuka Amarasuriya](https://gayanukaa.github.io/)


In [ ]:
import torch
import torch.nn as nn
import json
from datasets import load_dataset
from transformers import AutoProcessor, AutoModel, AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pycocoevalcap

## Configurations


In [3]:
# Model selection - uncomment one
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
# MODEL_NAME = "microsoft/Phi-3.5-vision-instruct"
# MODEL_NAME = "mistralai/Pixtral-12B-2409"
# MODEL_NAME = "google/gemma-2-9b-it"  # Text-only model

DATASET_NAME = "arampacha/rsicd"
MAX_LENGTH = 128
PROMPT_LENGTH = 4
BATCH_SIZE = 1
NUM_TRAIN_EPOCHS = 3
OUTPUT_DIR = "./manual_prompt_output"
device = "cuda" if torch.cuda.is_available() else "cpu"

## Universal VLM Prompt Tuning Implementation


In [ ]:
class UniversalVLMPromptTuning(nn.Module):
    def __init__(self, model, num_tokens=4):
        super().__init__()
        self.model = model
        self.num_tokens = num_tokens

        # Determine model type and get embeddings
        self.model_type = self._determine_model_type()
        self.embedding_dim = self._get_embedding_dim()

        # Initialize soft prompts on the same device as the model
        device = next(model.parameters()).device
        self.soft_prompts = nn.Parameter(torch.randn(num_tokens, self.embedding_dim, device=device))
        nn.init.normal_(self.soft_prompts, std=0.02)

        # Freeze base model parameters
        for param in self.model.parameters():
            param.requires_grad = False

    def _determine_model_type(self):
        model_name = self.model.__class__.__name__.lower()
        if "gemma" in model_name:
            return "gemma"
        elif "qwen" in model_name:
            return "qwen"
        elif "pixtral" in model_name:
            return "pixtral"
        elif "phi" in model_name:
            return "phi"
        else:
            return "unknown"

    def _get_embedding_dim(self):
        if hasattr(self.model, 'get_input_embeddings'):
            return self.model.get_input_embeddings().weight.shape[-1]
        elif hasattr(self.model, 'model') and hasattr(self.model.model, 'embed_tokens'):
            return self.model.model.embed_tokens.weight.shape[-1]
        else:
            return 768  # Default

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        """Enable gradient checkpointing for the base model if supported"""
        if hasattr(self.model, 'gradient_checkpointing_enable'):
            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs)

    def gradient_checkpointing_disable(self):
        """Disable gradient checkpointing for the base model if supported"""
        if hasattr(self.model, 'gradient_checkpointing_disable'):
            self.model.gradient_checkpointing_disable()

    def forward(self, input_ids=None, pixel_values=None, attention_mask=None, labels=None, **kwargs):
        if self.model_type == "gemma":
            # Text-only model
            if input_ids is not None:
                inputs_embeds = self.model.get_input_embeddings()(input_ids)
                batch_size = inputs_embeds.shape[0]

                # Add soft prompts
                soft_prompts_expanded = self.soft_prompts.unsqueeze(0).expand(batch_size, -1, -1)
                inputs_embeds = torch.cat([soft_prompts_expanded, inputs_embeds], dim=1)

                # Update attention mask
                if attention_mask is not None:
                    prompt_mask = torch.ones(batch_size, self.num_tokens, device=attention_mask.device)
                    attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

                return self.model(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    labels=labels,
                    **kwargs
                )
            else:
                return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, **kwargs)
        else:
            # Vision-language model
            return self.model(
                input_ids=input_ids,
                pixel_values=pixel_values,
                attention_mask=attention_mask,
                labels=labels,
                **kwargs
            )

    def generate(self, input_ids=None, pixel_values=None, attention_mask=None, **kwargs):
        if self.model_type == "gemma":
            # Text-only generation
            if input_ids is not None:
                inputs_embeds = self.model.get_input_embeddings()(input_ids)
                batch_size = inputs_embeds.shape[0]

                # Add soft prompts
                soft_prompts_expanded = self.soft_prompts.unsqueeze(0).expand(batch_size, -1, -1)
                inputs_embeds = torch.cat([soft_prompts_expanded, inputs_embeds], dim=1)

                # Update attention mask
                if attention_mask is not None:
                    prompt_mask = torch.ones(batch_size, self.num_tokens, device=attention_mask.device)
                    attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

                return self.model.generate(
                    inputs_embeds=inputs_embeds,
                    attention_mask=attention_mask,
                    **kwargs
                )
            else:
                return self.model.generate(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        else:
            # Vision-language generation
            return self.model.generate(
                input_ids=input_ids,
                pixel_values=pixel_values,
                attention_mask=attention_mask,
                **kwargs
            )

    def print_trainable_parameters(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

## Model Creation Helper


In [21]:
def create_prompt_tuned_model(model_name, num_tokens=4):
    """Create a prompt-tuned model with universal VLM support"""
    print(f"Loading model: {model_name}")

    if "gemma" in model_name.lower():
        # Text-only model
        processor = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True
        )
    else:
        # Vision-language model
        processor = AutoProcessor.from_pretrained(model_name, use_fast=True)
        model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True
        )

    # Move base model to device first
    model = model.to(device)

    # Wrap with universal prompt tuning
    prompt_model = UniversalVLMPromptTuning(model, num_tokens)

    print(f"Model type detected: {prompt_model.model_type}")
    prompt_model.print_trainable_parameters()

    return processor, prompt_model

## Load Dataset


In [22]:
dataset = load_dataset(DATASET_NAME)

# Dataset splits as planned
train_dataset = dataset["train"].select(range(1000))  # 1000 images for training
eval_dataset = dataset["valid"].select(range(200))    # 200 images for evaluation
test_dataset = dataset["test"].select(range(10))      # 10 images for testing

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Features: {train_dataset.features}")

Training samples: 1000
Evaluation samples: 200
Test samples: 10
Features: {'filename': Value(dtype='string', id=None), 'captions': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'image': Image(mode=None, decode=True, id=None)}


## Load Model and Processor


In [23]:
processor, model = create_prompt_tuned_model(MODEL_NAME, PROMPT_LENGTH)

# Alternative model options:
# processor, model = create_prompt_tuned_model("microsoft/Phi-3.5-vision-instruct", PROMPT_LENGTH)
# processor, model = create_prompt_tuned_model("mistralai/Pixtral-12B-2409", PROMPT_LENGTH)
# processor, model = create_prompt_tuned_model("google/gemma-2-9b-it", PROMPT_LENGTH)

Loading model: Qwen/Qwen2.5-VL-7B-Instruct


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 44.45 GiB of which 786.00 MiB is free. Process 1713735 has 14.69 GiB memory in use. Process 1730241 has 14.69 GiB memory in use. Process 1754060 has 14.29 GiB memory in use. Of the allocated memory 13.96 GiB is allocated by PyTorch, and 78.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Preprocess Dataset


In [ ]:
def preprocess_for_vlm(example):
    """Preprocess for vision-language models"""
    image = example["image"]

    # Format as conversation
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Describe this satellite image."}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["captions"][0]}
            ]
        }
    ]

    # Apply chat template
    text = processor.apply_chat_template(messages, tokenize=False)

    # Process inputs
    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # For causal LM, labels are same as input_ids
    result = {
        "input_ids": inputs["input_ids"].squeeze(),
        "attention_mask": inputs["attention_mask"].squeeze(),
        "labels": inputs["input_ids"].squeeze()
    }

    # Add pixel_values if present
    if "pixel_values" in inputs:
        result["pixel_values"] = inputs["pixel_values"].squeeze()

    return result

def preprocess_for_text_only(example):
    """Preprocess for text-only models like Gemma"""
    # For text-only models, we can only use the caption
    prompt = "Describe this satellite image: "
    target = example["captions"][0]
    text = prompt + target

    inputs = processor(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

    return {
        "input_ids": inputs["input_ids"].squeeze(),
        "attention_mask": inputs["attention_mask"].squeeze(),
        "labels": inputs["input_ids"].squeeze()
    }

# Choose preprocessing based on model type
if model.model_type == "gemma":
    preprocess_func = preprocess_for_text_only
    # For text-only models, we only need these columns
    columns_to_keep = ["input_ids", "attention_mask", "labels"]
else:
    preprocess_func = preprocess_for_vlm
    # For VLM models, we need these columns
    columns_to_keep = ["input_ids", "attention_mask", "labels", "pixel_values"]

# Apply preprocessing
print("Preprocessing datasets...")
train_dataset = train_dataset.map(preprocess_func, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(preprocess_func, remove_columns=eval_dataset.column_names)
test_dataset = test_dataset.map(preprocess_func, remove_columns=test_dataset.column_names)

# Filter to only keep required columns
train_dataset = train_dataset.select_columns([col for col in columns_to_keep if col in train_dataset.column_names])
eval_dataset = eval_dataset.select_columns([col for col in columns_to_keep if col in eval_dataset.column_names])
test_dataset = test_dataset.select_columns([col for col in columns_to_keep if col in test_dataset.column_names])

# Set format
train_dataset.set_format(type="torch")
eval_dataset.set_format(type="torch")
test_dataset.set_format(type="torch")

print(f"Training dataset columns: {train_dataset.column_names}")
print(f"Sample from training dataset: {train_dataset[0].keys()}")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

## Training Arguments


In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=10,
    eval_steps=100,
    save_steps=200,
    save_total_limit=1,
    eval_strategy="steps",
    fp16=True,
    report_to="none",
    remove_unused_columns=True,  # Enable to remove unused columns
    dataloader_pin_memory=False,
    gradient_checkpointing=False,  # Disabled for custom model compatibility
    learning_rate=3e-2,  # Higher learning rate for prompt tuning
    warmup_steps=50,
    dataloader_drop_last=True,  # Drop last incomplete batch
)

## Evaluation Metrics


In [13]:
# Initialize evaluation metrics
cider_scorer = Cider()
spice_scorer = Spice()

def compute_cosine_similarity(predictions, references):
    """Compute cosine similarity between predictions and references"""
    from sklearn.feature_extraction.text import TfidfVectorizer

    vectorizer = TfidfVectorizer()
    all_texts = predictions + [ref[0] for ref in references]

    try:
        tfidf_matrix = vectorizer.fit_transform(all_texts)
        pred_vectors = tfidf_matrix[:len(predictions)]
        ref_vectors = tfidf_matrix[len(predictions):]

        similarities = []
        for i in range(len(predictions)):
            sim = cosine_similarity(pred_vectors[i], ref_vectors[i])[0][0]
            similarities.append(sim)

        return np.mean(similarities)
    except:
        return 0.0

def compute_all_metrics(predictions, references):
    """Compute evaluation metrics using pycocoevalcap"""
    # Format data for pycocoevalcap (requires dict format)
    gts = {}  # ground truth
    res = {}  # results

    for i, (pred, ref_list) in enumerate(zip(predictions, references)):
        gts[i] = ref_list
        res[i] = [pred]

    # Compute metrics
    results = {}

    try:
        # CIDEr (TF-IDF + cosine)
        cider_score, _ = cider_scorer.compute_score(gts, res)
        results['CIDEr'] = cider_score
    except Exception as e:
        print(f"CIDEr calculation failed: {e}")
        results['CIDEr'] = 0.0

    try:
        # SPICE (F1 with scene graphs)
        spice_score, _ = spice_scorer.compute_score(gts, res)
        results['SPICE'] = spice_score
    except Exception as e:
        print(f"SPICE calculation failed: {e}")
        results['SPICE'] = 0.0

    # Cosine similarity
    results['Cosine_Similarity'] = compute_cosine_similarity(predictions, references)

    return results

def compute_metrics(eval_preds):
    """Compute metrics during training"""
    logits, labels = eval_preds
    predictions = torch.argmax(torch.tensor(logits), dim=-1)

    # Decode predictions and labels
    decoded_preds = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Clean predictions (extract generated part)
    cleaned_preds = []
    for pred in decoded_preds:
        if "assistant" in pred:
            cleaned_pred = pred.split("assistant")[-1].strip()
        else:
            cleaned_pred = pred.strip()
        cleaned_preds.append(cleaned_pred)

    # Compute subset of metrics for training (faster)
    try:
        gts = {i: [ref] for i, ref in enumerate(decoded_labels)}
        res = {i: [pred] for i, pred in enumerate(cleaned_preds)}

        cider_score, _ = cider_scorer.compute_score(gts, res)
        cosine_sim = compute_cosine_similarity(cleaned_preds, [[l] for l in decoded_labels])

        return {
            "CIDEr": cider_score,
            "Cosine_Similarity": cosine_sim
        }
    except:
        return {
            "CIDEr": 0.0,
            "Cosine_Similarity": 0.0
        }

Progress: 384.5M / 384.5M (100.0%)
Extracting stanford-corenlp-3.6.0 ...
Done.


## Train the Model


In [ ]:
from transformers import DataCollatorForLanguageModeling

# Custom trainer for manual prompt tuning
class PromptTuningTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Forward pass through our custom model
        outputs = model(**inputs)

        # Handle different output types
        if hasattr(outputs, 'loss') and outputs.loss is not None:
            loss = outputs.loss
        elif hasattr(outputs, 'logits'):
            # Manual loss calculation for models without built-in loss
            labels = inputs.get("labels")
            if labels is not None:
                shift_logits = outputs.logits[..., :-1, :].contiguous()
                shift_labels = labels[..., 1:].contiguous()
                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            else:
                loss = torch.tensor(0.0, requires_grad=True, device=next(model.parameters()).device)
        else:
            loss = torch.tensor(0.0, requires_grad=True, device=next(model.parameters()).device)

        return (loss, outputs) if return_outputs else loss

# Create a custom data collator for our specific needs
class VLMDataCollator:
    def __init__(self, tokenizer, model_type="vlm"):
        self.tokenizer = tokenizer
        self.model_type = model_type

    def __call__(self, features):
        # Extract all the keys from the first feature
        batch = {}

        # Handle input_ids, attention_mask, labels
        for key in ["input_ids", "attention_mask", "labels"]:
            if key in features[0]:
                batch[key] = torch.stack([f[key] for f in features])

        # Handle pixel_values for VLM models
        if "pixel_values" in features[0]:
            batch["pixel_values"] = torch.stack([f["pixel_values"] for f in features])

        return batch

# Create data collator
if model.model_type == "gemma":
    data_collator = VLMDataCollator(processor, model_type="text")
else:
    data_collator = VLMDataCollator(processor, model_type="vlm")

# Create trainer (model is already on correct device)
trainer = PromptTuningTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor.tokenizer if hasattr(processor, 'tokenizer') else processor,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

TypeError: Module.to_empty() takes 1 positional argument but 2 were given

## Test on Multiple Samples


In [ ]:
# Test on the 10 test samples
predictions = []
references = []
inference_times = []
vram_usage = []

# Get original test dataset for references
original_test = dataset["test"].select(range(10))

for i in range(len(original_test)):
    original_sample = original_test[i]

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    # Prepare inference input
    if model.model_type == "gemma":
        # Text-only inference
        prompt = "Describe this satellite image: "
        inputs = processor(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=64,
                temperature=0.7,
                do_sample=True
            )
    else:
        # Vision-language inference
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": original_sample["image"]},
                    {"type": "text", "text": "Describe this satellite image."}
                ]
            }
        ]

        input_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=input_text, images=original_sample["image"], return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                pixel_values=inputs.get("pixel_values"),
                max_new_tokens=64,
                temperature=0.7,
                do_sample=True
            )

    end_time = time.time()

    # Decode output
    generated_text = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract generated part
    if "assistant" in generated_text:
        generated_part = generated_text.split("assistant")[-1].strip()
    else:
        generated_part = generated_text.strip()

    predictions.append(generated_part)
    references.append([original_sample["captions"][0]])
    inference_times.append(end_time - start_time)
    vram_usage.append(torch.cuda.max_memory_allocated() / 1e9)

    print(f"Sample {i+1}:")
    print(f"Generated: {generated_part}")
    print(f"Reference: {original_sample['captions'][0]}")
    print(f"Inference Time: {inference_times[-1]:.3f}s")
    print(f"Peak VRAM: {vram_usage[-1]:.2f}GB")
    print("-" * 50)

# Calculate comprehensive metrics using pycocoevalcap
print("\n=== Final Test Results ===")
print(f"Average Inference Time: {np.mean(inference_times):.3f}s (±{np.std(inference_times):.3f}s)")
print(f"Average VRAM Usage: {np.mean(vram_usage):.2f}GB (±{np.std(vram_usage):.2f}GB)")

print("\n=== Evaluation Metrics ===")
metrics = compute_all_metrics(predictions, references)
for metric_name, score in metrics.items():
    print(f"{metric_name}: {score:.4f}")

# Print detailed results table
print("\n=== Metrics Summary ===")
print(f"{'Metric':<15} {'Score':<10}")
print("-" * 25)
for metric_name, score in metrics.items():
    print(f"{metric_name:<15} {score:<10.4f}")

## Save Model


In [ ]:
# Save the soft prompts
torch.save({
    'soft_prompts': model.soft_prompts,
    'model_type': model.model_type,
    'num_tokens': model.num_tokens,
    'model_name': MODEL_NAME
}, f"{OUTPUT_DIR}/soft_prompts.pt")

print(f"Soft prompts saved to {OUTPUT_DIR}/soft_prompts.pt")
print(f"Model type: {model.model_type}")
print(f"Virtual tokens: {model.num_tokens}")

## Load and Test Saved Model


In [ ]:
# Example of loading saved soft prompts
def load_soft_prompts(checkpoint_path, base_model):
    checkpoint = torch.load(checkpoint_path)

    # Create new prompt tuning model
    prompt_model = UniversalVLMPromptTuning(
        base_model,
        num_tokens=checkpoint['num_tokens']
    )

    # Load the trained soft prompts
    prompt_model.soft_prompts.data = checkpoint['soft_prompts']

    print(f"Loaded soft prompts for {checkpoint['model_type']} model")
    return prompt_model

# To load later:
# processor, base_model = create_prompt_tuned_model(MODEL_NAME, PROMPT_LENGTH)
# loaded_model = load_soft_prompts(f"{OUTPUT_DIR}/soft_prompts.pt", base_model.model)

In [ ]:
# Example of loading saved soft prompts
def load_soft_prompts(checkpoint_path, base_model):
    checkpoint = torch.load(checkpoint_path)

    # Create new prompt tuning model
    prompt_model = UniversalVLMPromptTuning(
        base_model,
        num_virtual_tokens=checkpoint['num_virtual_tokens']
    )

    # Load the trained soft prompts
    prompt_model.soft_prompts.data = checkpoint['soft_prompts']

    print(f"Loaded soft prompts for {checkpoint['model_type']} model")
    return prompt_model

# To load later:
# processor, base_model = create_prompt_tuned_model(MODEL_NAME, PROMPT_LENGTH)
# loaded_model = load_soft_prompts(f"{OUTPUT_DIR}/soft_prompts.pt", base_model.base_model)

# Evaluation with metrics
def evaluate_model(model, processor, test_dataset, device):
    model.eval()

    results = {
        'generated_captions': [],
        'reference_captions': [],
        'inference_times': [],
        'vram_usage': []
    }

    print(f"Evaluating on {len(test_dataset)} samples...")

    for i, sample in enumerate(test_dataset):
        # Measure inference time
        start_time = time.time()

        # Clear cache before inference
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            initial_memory = torch.cuda.memory_allocated(device)

        with torch.no_grad():
            if "gemma" in model.model_type:
                # Text-only model
                prompt = f"Describe this image: {sample['captions'][0]}"
                inputs = processor(prompt, return_tensors="pt", padding=True, truncation=True)

                # Ensure inputs are on the correct device
                inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

                outputs = model.generate(
                    **inputs,
                    max_new_tokens=50,
                    do_sample=False,
                    pad_token_id=processor.eos_token_id
                )

                generated_text = processor.decode(outputs[0], skip_special_tokens=True)
                generated_text = generated_text.replace(prompt, "").strip()
            else:
                # Vision-language model
                messages = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": sample["image"]},
                            {"type": "text", "text": "Describe this satellite image."}
                        ]
                    }
                ]

                input_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = processor(
                    text=input_text,
                    images=sample["image"],
                    return_tensors="pt",
                    padding=True
                )

                # Ensure inputs are on the correct device
                inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

                outputs = model.generate(
                    **inputs,
                    max_new_tokens=50,
                    do_sample=False
                )

                generated_text = processor.decode(outputs[0], skip_special_tokens=True)
                # Extract generated part
                if "assistant" in generated_text:
                    generated_text = generated_text.split("assistant")[-1].strip()
                else:
                    generated_text = generated_text.strip()

        # Measure time and memory
        end_time = time.time()
        inference_time = end_time - start_time

        if torch.cuda.is_available():
            peak_memory = torch.cuda.max_memory_allocated(device)
            vram_usage = (peak_memory - initial_memory) / 1024**2  # MB
        else:
            vram_usage = 0

        results['generated_captions'].append(generated_text)
        results['reference_captions'].append(sample['captions'][0])
        results['inference_times'].append(inference_time)
        results['vram_usage'].append(vram_usage)

        if i % 5 == 0:
            print(f"Sample {i+1}/{len(test_dataset)} - Time: {inference_time:.3f}s, VRAM: {vram_usage:.1f}MB")

    return results

# Run evaluation on test dataset
print("Running full evaluation...")
eval_results = evaluate_model(model, processor, test_dataset, device)

# Calculate metrics
print("\nCalculating metrics...")
references = [[ref] for ref in eval_results['reference_captions']]
hypotheses = eval_results['generated_captions']

# Format for pycocoevalcap
gts = {i: [ref] for i, ref in enumerate(eval_results['reference_captions'])}
res = {i: [hyp] for i, hyp in enumerate(eval_results['generated_captions'])}

# CIDER and SPICE using pycocoevalcap
try:
    cider_score, _ = cider_scorer.compute_score(gts, res)
    print(f"CIDEr: {cider_score:.4f}")
except Exception as e:
    print(f"CIDEr calculation failed: {e}")
    cider_score = 0.0

try:
    spice_score, _ = spice_scorer.compute_score(gts, res)
    print(f"SPICE: {spice_score:.4f}")
except Exception as e:
    print(f"SPICE calculation failed: {e}")
    spice_score = 0.0

# Cosine similarity
avg_cosine_similarity = compute_cosine_similarity(
    eval_results['generated_captions'],
    [[ref] for ref in eval_results['reference_captions']]
)

# Performance metrics
avg_inference_time = np.mean(eval_results['inference_times'])
avg_vram_usage = np.mean(eval_results['vram_usage'])

print(f"\nEvaluation Results:")
print(f"CIDEr: {cider_score:.4f}")
print(f"SPICE: {spice_score:.4f}")
print(f"Cosine Similarity: {avg_cosine_similarity:.4f}")
print(f"Avg Inference Time: {avg_inference_time:.3f}s")
print(f"Avg VRAM Usage: {avg_vram_usage:.1f}MB")

# Save results
results_dict = {
    'cider': cider_score,
    'spice': spice_score,
    'cosine_similarity': avg_cosine_similarity,
    'inference_time': avg_inference_time,
    'vram_usage': avg_vram_usage,
    'generated_captions': eval_results['generated_captions'],
    'reference_captions': eval_results['reference_captions']
}

with open(f'evaluation_results_{MODEL_NAME.replace("/", "_")}.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

print(f"\nResults saved to evaluation_results_{MODEL_NAME.replace('/', '_')}.json")